# TNC Binder Demo

This notebook is the lightweight Binder version of the project. It uses a sampled dataframe, trains only `ExtraTreesRegressor`, and keeps the default output set small so Binder can run it more reliably.

For the full workflow, use the local script or the fuller notebook in this repo.

## 1. Imports

In [ ]:
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "scripts").exists() and (repo_root.parent / "scripts").exists():
    repo_root = repo_root.parent
sys.path.append(str(repo_root / "scripts"))

from Feature_Engineering import Feature_Engineering
from ML_Modeling import ML_Modeling

pd.set_option("display.max_columns", None)

## 2. Paths

In [ ]:
data_dir = repo_root / "ML_Modeling_Files" / "TIFF_Files_For_Model" / "Chimney_Springs_P-dry"

predictor_paths = [
    data_dir / "CC.tif",
    data_dir / "CH.tif",
    data_dir / "East.tif",
    data_dir / "North.tif",
    data_dir / "PAG.tif",
    data_dir / "SFI.tif",
    data_dir / "SVF.tif",
    data_dir / "WSI.tif",
]
target_path = data_dir / "SCD_2020_SnowPALM_Map.tif"
target_variable = target_path.stem
output_dir = repo_root / "binder_outputs"
output_dir.mkdir(exist_ok=True)

all_paths = predictor_paths + [target_path]
missing_files = [path for path in all_paths if not path.exists()]

if missing_files:
    print("Missing files:")
    for path in missing_files:
        print(f"- {path}")
else:
    print("All expected GeoTIFF files are present.")

## 3. Sampled dataframe

In [ ]:
feature_engineering = Feature_Engineering()
sample_rows = 10000

if missing_files:
    print("Add the missing TIFFs before running the rest of the notebook.")
else:
    binder_df = feature_engineering.preprocess_data(
        [str(path) for path in predictor_paths],
        str(target_path),
        sample_rows=sample_rows,
    )
    display(binder_df.head())
    print(binder_df.shape)

## 4. Correlation matrix

In [ ]:
if missing_files:
    print("Skipping visualization because files are missing.")
else:
    feature_engineering.create_correlation_matrix(
        binder_df,
        target_path.stem,
        output_dir=str(output_dir),
    )
    print("Correlation matrix saved in binder_outputs/.")

## 5. Optional pair plot

Run this cell manually only if the sampled dataframe is small enough and you want the heavier pair plot.

In [ ]:
show_pair_plot = False

if missing_files:
    print("Skipping pair plot because files are missing.")
elif show_pair_plot:
    feature_engineering.create_pair_plot(
        binder_df,
        target_path.stem,
        output_dir=str(output_dir),
        sample_rows=min(sample_rows, 3000),
    )
    print("Pair plot saved in binder_outputs/.")
else:
    print("Pair plot is disabled by default.")

## 6. Extra Trees only

In [ ]:
ml_modeling = ML_Modeling()

if missing_files:
    print("Skipping model training because files are missing.")
else:
    ml_modeling.create_ML_Model(
        [str(path) for path in predictor_paths],
        str(target_path),
        target_variable,
        model_types=["Extra Trees"],
        sample_rows=sample_rows,
        output_dir=str(output_dir),
        save_all_metrics=True,
    )
    print("Extra Trees model saved in binder_outputs/.")

## 7. Saved metrics

In [ ]:
metrics_path = output_dir / "et_metrics_test.joblib"

if metrics_path.exists():
    ml_modeling.read_joblib_metric(metrics_path)
else:
    print("No metrics file yet.")

## 8. Review outputs

This section renders the files saved into `binder_outputs/` so the demo stays visible inside Binder instead of only writing artifacts to disk.

In [ ]:
from IPython.display import Image, display

image_groups = [
    ("Correlation matrix", [output_dir / f"{target_path.stem}_correlation_matrix.png"]),
    ("Model diagnostics", [
        output_dir / "et_predicted_versus_actual.png",
        output_dir / "et_residual_histogram.png",
        output_dir / "et_residual_versus_predicted.png",
    ]),
    ("Map outputs", [
        output_dir / f"{target_path.stem}_actual.png",
        output_dir / f"{target_path.stem}_predicted.png",
    ]),
    ("Optional pair plot", [output_dir / f"{target_path.stem}_pair_plot.png"]),
]

for title, paths in image_groups:
    available = [path for path in paths if path.exists()]
    if not available:
        continue
    print(f"\n{title}")
    for path in available:
        display(Image(filename=str(path)))

metric_files = sorted(output_dir.glob("*_metrics_*.joblib"))
if metric_files:
    print("\nSaved metrics")
    for path in metric_files:
        print(path.name)
        ml_modeling.read_joblib_metric(path)
else:
    print("No saved metrics found yet.")

Binder is for the light path. If you want the full multi-model workflow and the denser plots, run the local notebook or the `User.py` script on your machine.